<a href="https://colab.research.google.com/github/annagiacometti/poster-ocr-llm-structured-extraction/blob/main/poster_ocr_llm_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estrazione strutturata di metadati da locandine di eventi tramite OCR e LLM
Pipeline sperimentale a due stadi per l'estrazione automatica di informazioni chiave (`festival_name`, `city_name`, `start_date`, `end_date`, `duration_days`) a partire da immagini grafiche eterogenee. Il flusso combina riconoscimento ottico dei caratteri (**EasyOCR**), normalizzazione semantica in formato JSON tramite modello linguistico (**Qwen/Qwen2.5-3B-Instruct**), post-processing deterministico in Python e valutazione rispetto a valori di riferimento (*ground truth*).

## 1. Configurazione dell'ambiente e accesso ai dati
Installazione delle librerie necessarie (`easyocr`, `transformers`, `accelerate`, `openpyxl`), importazione dei moduli Python e collegamento a Google Drive per accedere alla cartella contenente il campione di 20 locandine.

In [5]:
!pip install -q easyocr transformers accelerate openpyxl

import os
import json
import torch
import easyocr
import pandas as pd
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer

In [6]:
CARTELLA_LOCANDINE = "/content/drive/MyDrive/locandine_festival"

## 2. Inizializzazione dei modelli (OCR + LLM)
Caricamento su GPU dei due componenti della pipeline:
1. **EasyOCR** (lingue italiano e inglese) per l'estrazione del testo grezzo dalle immagini.
2. **Qwen/Qwen2.5-3B-Instruct** in precisione `float16` per interpretare il testo non strutturato e mapparlo in formato JSON.

In [7]:
print("Caricamento EasyOCR (IT/EN)...")
reader = easyocr.Reader(['it', 'en'], gpu=True)

print("Caricamento Qwen/Qwen2.5-3B-Instruct...")
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Modelli pronti.")

Caricamento EasyOCR (IT/EN)...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteCaricamento Qwen/Qwen2.5-3B-Instruct...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelli pronti.


## 3. Schema JSON e funzione di estrazione strutturata
Definizione del prompt di sistema e dello schema a 5 chiavi (`festival_name`, `city_name`, `start_date`, `end_date`, `duration_days`). Per limitare le allucinazioni su locandine con informazioni incomplete, il modello utilizza decodifica deterministica (`do_sample=False`) ed è istruito a restituire `null` per ogni campo non rilevabile nel testo OCR.

In [8]:
def estrai_metadati_da_ocr(testo_ocr):
    messages = [
        {
            "role": "system",
            "content": (
                "Sei un assistente specializzato nell'estrazione di metadati strutturati da testi OCR grezzi di locandine di festival fotografici. "
                "Restituisci ESCLUSIVAMENTE un oggetto JSON valido con queste 5 chiavi esatte:\n"
                "- 'festival_name': stringa con il nome principale del festival (o null se non chiaro).\n"
                "- 'city_name': stringa con la città o il luogo dell'evento (o null se assente).\n"
                "- 'start_date': stringa con la data di inizio nel formato 'YYYY-MM-DD' (usa 'XXXX-MM-DD' se l'anno non è scritto; usa null se non ci sono date).\n"
                "- 'end_date': stringa con la data di fine nel formato 'YYYY-MM-DD' (se l'evento dura un solo giorno, ripeti 'start_date'; usa null se non ci sono date).\n"
                "- 'duration_days': numero intero dei giorni complessivi di durata, calcolato INCLUDENDO sia il giorno di inizio che il giorno di fine (formula: (end_date - start_date) + 1; es. dal 6 all'8 ottobre = 3 giorni; un solo giorno = 1). Usa null se le date mancano.\n"
                "REGOLA FONDAMENTALE: Non inventare mai dati non presenti nel testo OCR. Se un'informazione manca, usa null."
            )
        },
        {
            "role": "user",
            "content": f"Testo OCR estratto dalla locandina:\n\"\"\"\n{testo_ocr}\n\"\"\""
        }
    ]

    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_input], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    output_ids = generated_ids[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(output_ids, skip_special_tokens=True).strip()

## 4. Esecuzione batch, parsing JSON e calcolo deterministico della durata
Elaborazione sequenziale delle 20 immagini:
* **Estrazione OCR:** i blocchi testuali rilevati vengono uniti tramite il separatore ` | ` per preservare la distinzione tra elementi grafici diversi.
* **Strutturazione LLM:** generazione e validazione sintattica dell'output JSON.
* **Post-processing Python:** calcolo deterministico della durata inclusiva (`(end_date - start_date) + 1`) a partire dalle date estratte, affiancato alla stima diretta del modello per confronto.

In [9]:
estensioni_valide = ('.jpg', '.jpeg', '.png', '.webp')
file_immagini = sorted([f for f in os.listdir(CARTELLA_LOCANDINE) if f.lower().endswith(estensioni_valide)])

print(f"Trovate {len(file_immagini)} immagini. Avvio pipeline...\n")

risultati_finali = []
json_validi = 0

for idx, nome_file in enumerate(file_immagini, 1):
    percorso_completo = os.path.join(CARTELLA_LOCANDINE, nome_file)
    poster_id = os.path.splitext(nome_file)[0]
    festival_id = poster_id.split("_")[0]

    # Fase A: Estrazione testo grezzo con EasyOCR
    blocchi_testo = reader.readtext(percorso_completo, detail=0)
    testo_ocr_pulito = " | ".join(blocchi_testo)

    # Fase B: Estrazione strutturata con Qwen2.5-3B
    output_grezzo = estrai_metadati_da_ocr(testo_ocr_pulito)
    json_str = output_grezzo.replace("```json", "").replace("```", "").strip()

    try:
        dati_parsed = json.loads(json_str)
        json_validi += 1
        stato_parsing = "OK"
    except json.JSONDecodeError:
        dati_parsed = {}
        stato_parsing = "ERRORE_JSON"

    risultati_finali.append({
        "poster_id": poster_id,
        "festival_id": festival_id,
        "file": nome_file,
        "stato_json": stato_parsing,
        "festival_name": dati_parsed.get("festival_name"),
        "city_name": dati_parsed.get("city_name"),
        "start_date": dati_parsed.get("start_date"),
        "end_date": dati_parsed.get("end_date"),
        "duration_days": dati_parsed.get("duration_days"),
        "testo_ocr_grezzo": testo_ocr_pulito
    })

    print(f"[{idx}/{len(file_immagini)}] {nome_file} -> {dati_parsed.get('festival_name')} | {dati_parsed.get('start_date')} -> {dati_parsed.get('end_date')} ({dati_parsed.get('duration_days')} gg)")

df_risultati = pd.DataFrame(risultati_finali)
print(f"\nValidità sintattica JSON: {json_validi}/{len(file_immagini)}")

from datetime import datetime
import numpy as np

def calcola_durata_inclusiva(start_s, end_s):
    if not start_s or not end_s or pd.isna(start_s) or pd.isna(end_s):
        return np.nan
    try:
        # Sostituisce eventuale anno mancante 'XXXX' con un anno bisestile di servizio per calcolare i giorni
        d1 = datetime.strptime(str(start_s).replace("XXXX", "2024"), "%Y-%m-%d")
        d2 = datetime.strptime(str(end_s).replace("XXXX", "2024"), "%Y-%m-%d")
        return (d2 - d1).days + 1
    except ValueError:
        return np.nan

# Mantiene la stima diretta dell'LLM e aggiunge il calcolo deterministico Python
df_risultati = df_risultati.rename(columns={"duration_days": "duration_days_llm"})
df_risultati["duration_days_python"] = [
    calcola_durata_inclusiva(s, e) for s, e in zip(df_risultati["start_date"], df_risultati["end_date"])
]

Trovate 20 immagini. Avvio pipeline...

[1/20] FEST000003.jpeg -> CENTRALEFESTIVALCoM | 2023-10-06 -> 2023-10-08 (3 gg)
[2/20] FEST000005.jpg -> WorlD Press PhoTo | 2025-10-26 -> 2025-10-27 (2 gg)
[3/20] FEST000008_1.jpg -> AMatter | of Taste | None -> None (None gg)
[4/20] FEST000008_2.webp -> YEASTPHOTOFESTIVAL.IT | 2024-10-06 -> 2024-11-03 (28 gg)
[5/20] FEST000012.jpg -> Zero | Negativo | 2026-06-01 -> 2026-06-30 (30 gg)
[6/20] FEST000014.jpg -> THE TIME | 2026-09-05 -> 2026-09-24 (19 gg)
[7/20] FEST000021_1.jpg -> FESTIVAL INTERNAZIONALE DI FOTOGRAFIA | 2026-11-16 -> 2026-11-17 (2 gg)
[8/20] FEST000021_2.jpg -> On The Move | 2024-07-11 -> 2024-11-03 (63 gg)
[9/20] FEST000052.jpg -> Photol | TPD Program 2026 | 2026-10-22 -> 2026-10-25 (3 gg)
[10/20] FEST000065.png -> Photo Festival | 2022-07-29 -> 2022-09-10 (32 gg)
[11/20] FEST000067.jpg -> RAISE YOUR ZINE! Fanzine Magazine & Photobook Festival | 2026-10-06 -> 2026-10-08 (3 gg)
[12/20] FEST000069.jpg -> STRFESHAX | 2024-04-26 -> 2

## 5. Esportazione dei dataset estratti
Salvataggio dei risultati completi (metadati estratti, ID univoci e testo OCR grezzo) nei formati `.csv` e `.xlsx` per la successiva fase di verifica manuale e confronto.

In [12]:
df_risultati.to_csv("festival_posters_extracted_metadata.csv", sep=";", index=False, encoding="utf-8-sig")
df_risultati.to_excel("festival_posters_extracted_metadata.xlsx", index=False)

display(df_risultati[["file", "festival_name", "city_name", "start_date", "end_date", "duration_days_llm", "duration_days_python"]])

,file,festival_name,city_name,start_date,end_date,duration_days_llm,duration_days_python
0,FEST000003.jpeg,CENTRALEFESTIVALCoM,MILANO,2023-10-06,2023-10-08,3.0,3.0
1,FEST000005.jpg,WorlD Press PhoTo,LODI,2025-10-26,2025-10-27,2.0,2.0
2,FEST000008_1.jpg,AMatter | of Taste,Lecce,None,None,NaN,NaN
3,FEST000008_2.webp,YEASTPHOTOFESTIVAL.IT,Fahrfnhorst,2024-10-06,2024-11-03,28.0,29.0
4,FEST000012.jpg,Zero | Negativo,Altamura,2026-06-01,2026-06-30,30.0,30.0
5,FEST000014.jpg,THE TIME,None,2026-09-05,2026-09-24,19.0,20.0
6,FEST000021_1.jpg,FESTIVAL INTERNAZIONALE DI FOTOGRAFIA,CORTONA,2026-11-16,2026-11-17,2.0,2.0
7,FEST000021_2.jpg,On The Move,Cortona,2024-07-11,2024-11-03,63.0,116.0
8,FEST000052.jpg,Photol | TPD Program 2026,Trieste,2026-10-22,2026-10-25,3.0,4.0
9,FEST000065.png,Photo Festival,Fiumefreddo,2022-07-29,2022-09-10,32.0,44.0


## 6. Valutazione quantitativa e confronto con la Ground Truth
Incrocio 1-a-1 tra i risultati della pipeline e il dataset di riferimento annotato manualmente (`ground_truth_locandine.xlsx`) tramite la chiave univoca `poster_id`. Calcolo dell'accuratezza per singola variabile e confronto tra la durata stimata direttamente dall'LLM e quella ricalcolata deterministicamente in Python.